In [33]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### 4.- ANALISIS DE COHORTE

##### 4-1 Introduccion a las Cohortes

<span style="font-size: 12px;">

Agrupamiento generacional: Cual es caracteristica para agrupar gente en la cohorte, por ejemplo: fecha de nacimiento, primer compra de un producto, nacionalidad, género, sexo. No debe cambiar a lo largo del estudio.

Temporalidad: Tiempo en que se estará siguiendo al grupo.

Métrica agregada: Cómo mediremos a las personas de la cohorte, pueden ser ventas, ticket promedio, calificaciones, asistencia a clase, tasa de clicks, etc.


Temas:

Análisis de retención: Si cada miembro de una generación está presenta en una fecha en particular de nuestro estudio.

Análisis de superviviencia: Cuanta gente de una generación sobrevive después de cierto tiempo.

Análisis de retorno: Si una acción ha sucedido más de n veces en un periodo de tiempo dado. Por ejemplo, cuantas veces nos compra un cliente en un periodo de tiempo.

Análisis Acumulativos: A lo largo de la vida del cliente, cuento dinero ha gastado con nosotros.

</span>


##### 4-2 Introducción a Análisis de Retencion

<span style="font-size: 12px;">

Qué tanta gente se queda con nosotros. Ver el tamaño de la cohorte a través del tiempo, o bien, que tanto dinero gasta la gente a traves del tiempo. 

Buscamos expresar como un %. Al inicio se tiene un 100%, luego puede aumentar o disminuir.

Se puede considerar el número de personas o gasto de dinero a través del tiempo.

</span>

##### 4-3 SQL para curva basica de Retencion

In [3]:
query="""
SELECT
     * 
FROM
     cohort.supertienda
LIMIT
     10;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,venta_id,Categoria,Ciudad,Pais/Región,Nombre_del_Cliente,Proveedor,Fecha_de_Orden,ID_Orden,Codigo_Postal,Nombre_de_Producto,Region,Segmento,Fecha_de_envio,Estado/Provincia,Descuento,Ganancia,Cantidad,Ventas
0,1,Suministros de Oficina,Houston,United States,Darren Powers,Message Book,2019-01-03,US-2019-103800,77095,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Central,Consumidor,2019-01-07,Texas,0.2,5.5512,2,16.448
1,2,Suministros de Oficina,Naperville,United States,Phillina Ober,GBC,2019-01-04,US-2019-112326,60540,GBC Standard Plastic Binding Systems Combs,Central,Home Office,2019-01-08,Illinois,0.8,-5.4870,2,3.540
2,3,Suministros de Oficina,Naperville,United States,Phillina Ober,Avery,2019-01-04,US-2019-112326,60540,Avery 508,Central,Home Office,2019-01-08,Illinois,0.2,4.2717,3,11.784
3,4,Suministros de Oficina,Naperville,United States,Phillina Ober,SAFCO,2019-01-04,US-2019-112326,60540,SAFCO Boltless Steel Shelving,Central,Home Office,2019-01-08,Illinois,0.2,-64.7748,3,272.736
4,5,Suministros de Oficina,Philadelphia,United States,Mick Brown,Avery,2019-01-05,US-2019-141817,19143,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Este,Consumidor,2019-01-12,Pennsylvania,0.2,4.8840,3,19.536
5,6,Muebles,Henderson,United States,Maria Etezadi,Global,2019-01-06,US-2019-167199,42420,Global Deluxe High-Back Manager's Chair,Sur,Home Office,2019-01-10,Kentucky,0.0,746.4078,9,2573.820
6,7,Suministros de Oficina,Henderson,United States,Maria Etezadi,Rogers,2019-01-06,US-2019-167199,42420,Rogers Handheld Barrel Pencil Sharpener,Sur,Home Office,2019-01-10,Kentucky,0.0,1.4796,2,5.480
7,8,Suministros de Oficina,Athens,United States,Jack O'Briant,Dixon,2019-01-06,US-2019-106054,30605,"Dixon Prang Watercolor Pencils, 10-Color Set w...",Sur,Corporativo,2019-01-07,Georgia,0.0,5.2398,3,12.780
8,9,Suministros de Oficina,Henderson,United States,Maria Etezadi,Ibico,2019-01-06,US-2019-167199,42420,Ibico Hi-Tech Manual Binding System,Sur,Home Office,2019-01-10,Kentucky,0.0,274.4910,2,609.980
9,10,Suministros de Oficina,Henderson,United States,Maria Etezadi,Alliance,2019-01-06,US-2019-167199,42420,"Alliance Super-Size Bands, Assorted Sizes",Sur,Home Office,2019-01-10,Kentucky,0.0,0.3112,4,31.120


In [4]:
# Convertir las columnas Fecha_deOrden y Fecha_de_envio a tipo date

query="""
ALTER TABLE cohort.supertienda
MODIFY COLUMN Fecha_de_Orden DATE,
MODIFY COLUMN Fecha_de_envio DATE;
"""
with engine.begin() as conn:
    resultado = conn.execute(text(query))

print(f"Filas actualizadas: {resultado.rowcount}")

Filas actualizadas: 0


In [6]:
# Obtener la fecha en que cada cliente realizó su primera orden

query="""
select
    nombre_del_cliente,
    min(fecha_de_orden) as fecha_primera_orden
from
    cohort.supertienda
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,nombre_del_cliente,fecha_primera_orden
0,Darren Powers,2019-01-03
1,Phillina Ober,2019-01-04
2,Mick Brown,2019-01-05
3,Maria Etezadi,2019-01-06
4,Jack O'Briant,2019-01-06
5,Lycoris Saunders,2019-01-06
6,Vivek Sundaresam,2019-01-07
7,Melanie Seite,2019-01-09
8,Anthony Jacobs,2019-01-10
9,Seth Vernon,2019-01-11


In [7]:
# Obtener cuantos clientes compraron, cuantos compraron 1 mes después de su primera compra, 2 meses después de su primera compra, etc

query="""
select
    timestampdiff(MONTH, A.fecha_primera_orden, B.fecha_de_orden) as periodo,
    count(distinct B.nombre_del_cliente) as nro_clientes
from
    (
    select
        nombre_del_cliente,
        min(fecha_de_orden) as fecha_primera_orden
    from
        cohort.supertienda
    group by 1
    ) AS A
    inner join
    cohort.supertienda as B on A.nombre_del_cliente = B.nombre_del_cliente
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,periodo,nro_clientes
0,0,800
1,1,69
2,2,74
3,3,71
4,4,61
5,5,89
6,6,74
7,7,87
8,8,81
9,9,75


In [8]:
# Con la consulta anterior, obtener el porcentaje de cada periodo respecto al periodo 0

query="""
select
    periodo,
    nro_clientes,
    first_value(nro_clientes) over (order by periodo) as clientes_iniciales,
    (100 * nro_clientes / first_value(nro_clientes) over (order by periodo) ) as porcentaje_retencion
from
    (
    select
        timestampdiff(MONTH, A.fecha_primera_orden, B.fecha_de_orden) as periodo,
        count(distinct B.nombre_del_cliente ) as nro_clientes
    from 
        (
        select
            nombre_del_cliente,
            min(fecha_de_orden) as fecha_primera_orden
        from
            cohort.supertienda
        group by 	1
        ) AS A 
        inner join 
        cohort.supertienda as B on A.nombre_del_cliente = B.nombre_del_cliente
    group by
        1
    ) AS Z;
"""
df = pd.read_sql(text(query),engine)
df.head(10)


,periodo,nro_clientes,clientes_iniciales,porcentaje_retencion
0,0,800,800,100.000
1,1,69,800,8.625
2,2,74,800,9.250
3,3,71,800,8.875
4,4,61,800,7.625
5,5,89,800,11.125
6,6,74,800,9.250
7,7,87,800,10.875
8,8,81,800,10.125
9,9,75,800,9.375


##### 4-4 Analisis de Retencion Legisladores


In [9]:
query = """
SELECT
    * 
FROM
    cohort.legisladoresusa
LIMIT
    5;
"""
df = pd.read_sql(text(query),engine)
df.head()

,IDLegislador,DipSen,InicioPeriodo,FinPeriodo,Estado
0,B000944,rep,1993-01-05,1995-01-03,OH
1,C000127,rep,1993-01-05,1995-01-03,WA
2,C000141,rep,1987-01-06,1989-01-03,MD
3,C000174,rep,1983-01-03,1985-01-03,DE
4,C001070,sen,2007-01-04,2013-01-03,PA


In [11]:
# Explorar el contenido de la tabla de fechas

query="""
SELECT
    * 
FROM
    cohort.tablafechas
"""
df = pd.read_sql(text(query),engine)
df.head(10)


,Year,Mes,Dia,Dias en Year
0,2020,12,31,366
1,2019,12,31,365
2,2018,12,31,365
3,2017,12,31,365
4,2016,12,31,366
5,2015,12,31,365
6,2014,12,31,365
7,2013,12,31,365
8,2012,12,31,366
9,2011,12,31,365


In [25]:
# Agregar una columna de fecha 
query_alter="""
ALTER TABLE cohort.tablafechas ADD COLUMN Fecha DATE;
"""

# Definir el contenido de la columna Fecha
query_update="""
UPDATE cohort.tablafechas SET cohort.tablafechas.Fecha = concat(Year,"-",Mes,"-",Dia);
"""

# Ejecutar las consultas de alteración y actualización de la tabla
with engine.begin() as conn:
    conn.execute(text(query_alter))                  # Ejecutar el query de alteración de la tabla
    resultado = conn.execute(text(query_update))     # Ejecutar el query de actualización de la tabla, para saber cuantas filas fueron actualizadas, se puede usar resultado.rowcount

print(f"Filas actualizadas: {resultado.rowcount}")

Filas actualizadas: 251


In [ ]:
# En caso de necesitar eliminar la columna Fecha, se puede ejecutar la siguiente consulta:

query = """
ALTER TABLE cohort.tablafechas
DROP COLUMN Fecha;
"""

with engine.begin() as conn:
    conn.execute(text(query))

In [ ]:
# Explorar el contenido actualizado de la tabla de fechas

query="""
SELECT
    * 
FROM
    cohort.tablafechas
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,Year,Mes,Dia,Dias en Year,Fecha
0,2020,12,31,366,2020-12-31
1,2019,12,31,365,2019-12-31
2,2018,12,31,365,2018-12-31
3,2017,12,31,365,2017-12-31
4,2016,12,31,366,2016-12-31
5,2015,12,31,365,2015-12-31
6,2014,12,31,365,2014-12-31
7,2013,12,31,365,2013-12-31
8,2012,12,31,366,2012-12-31
9,2011,12,31,365,2011-12-31


In [29]:
# Cuando inicia cada legislador su periodo

query="""
SELECT
    IDLegislador,
    MIN(InicioPeriodo) as Primer_Periodo
FROM
    cohort.legisladoresusa
GROUP BY
    1
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,Primer_Periodo
0,B000944,1993-01-05
1,C000127,1993-01-05
2,C000141,1987-01-06
3,C000174,1983-01-03
4,C001070,2007-01-04
5,F000062,1992-11-10
6,F000469,2019-01-03
7,K000367,2007-01-04
8,M000639,1993-01-05
9,S000033,1991-01-03


In [30]:
# Agregar la fecha de primer periodo a la tabla original de legisladores

query="""
SELECT
    *
FROM
    (
    SELECT
        IDLegislador,
        MIN(InicioPeriodo) as Primer_Periodo
    FROM
        cohort.legisladoresusa
    GROUP BY
        1
    ) AS A 
    INNER JOIN
    cohort.legisladoresusa AS B ON A.IDLegislador=B.IDLegislador;
"""
pd = pd.read_sql(text(query), engine)
pd.head()

,IDLegislador,Primer_Periodo,IDLegislador,DipSen,InicioPeriodo,FinPeriodo,Estado
0,B000944,1993-01-05,B000944,rep,1993-01-05,1995-01-03,OH
1,C000127,1993-01-05,C000127,rep,1993-01-05,1995-01-03,WA
2,C000141,1987-01-06,C000141,rep,1987-01-06,1989-01-03,MD
3,C000174,1983-01-03,C000174,rep,1983-01-03,1985-01-03,DE
4,C001070,2007-01-04,C001070,sen,2007-01-04,2013-01-03,PA


In [37]:
# Usar la tabla de fechas para agregar las fechas intermedias entre InicioPeriodo y FinPeriodo y calcular el periodo.

query="""
SELECT
    A.IDLegislador,
    A.Primer_Periodo,
    B.InicioPeriodo,
    B.FinPeriodo,
    C.Fecha,
    COALESCE(timestampdiff(YEAR,A.primer_periodo,C.fecha),0) as Periodo
FROM
    (
    SELECT
        IDLegislador,
        MIN(InicioPeriodo) as Primer_Periodo
    FROM
        cohort.legisladoresusa
    GROUP BY
        1
    ) AS A 
    INNER JOIN 
    cohort.legisladoresusa AS B ON A.IDLegislador=B.IDLegislador
    LEFT JOIN 
    cohort.tablafechas AS C ON C.FECHA BETWEEN B.InicioPeriodo AND B.FInPeriodo
ORDER BY
    A.IDLegislador, B.InicioPeriodo, C.Fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,Primer_Periodo,InicioPeriodo,FinPeriodo,Fecha,Periodo
0,A000001,1951-01-03,1951-01-03,1953-01-03,1951-12-31,0
1,A000001,1951-01-03,1951-01-03,1953-01-03,1952-12-31,1
2,A000002,1947-01-03,1947-01-03,1949-01-03,1947-12-31,0
3,A000002,1947-01-03,1947-01-03,1949-01-03,1948-12-31,1
4,A000002,1947-01-03,1949-01-03,1951-01-03,1949-12-31,2
5,A000002,1947-01-03,1949-01-03,1951-01-03,1950-12-31,3
6,A000002,1947-01-03,1951-01-03,1953-01-03,1951-12-31,4
7,A000002,1947-01-03,1951-01-03,1953-01-03,1952-12-31,5
8,A000002,1947-01-03,1953-01-03,1955-01-03,1953-12-31,6
9,A000002,1947-01-03,1953-01-03,1955-01-03,1954-12-31,7


In [41]:
# Con la columna periodo calculada en el punto anterior, calcular el numero de legisladores en cada uno de los periodos */

query="""
SELECT
    Periodo,
    count(distinct IDLegislador) as numero_legisladores
FROM
    (
    SELECT
        A.IDLegislador,
        A.Primer_Periodo,
        B.InicioPeriodo,
        B.FinPeriodo,
        C.Fecha,
        COALESCE(timestampdiff(YEAR,A.primer_periodo,C.fecha),0) as Periodo
    FROM
        (
        SELECT
            IDLegislador,
            MIN(InicioPeriodo) as Primer_Periodo
        FROM
            cohort.legisladoresusa
        GROUP BY 1
        ) AS A 
        INNER JOIN 
        cohort.legisladoresusa AS B ON A.IDLegislador=B.IDLegislador
        LEFT JOIN 
        cohort.tablafechas AS C ON C.FECHA BETWEEN B.InicioPeriodo AND B.FInPeriodo
    ORDER BY A.IDLegislador, B.InicioPeriodo, C.Fecha
    ) AS M
GROUP BY 
    1
ORDER BY 
    1 ASC;
"""

df = pd.read_sql(text(query), engine)
df.head(10)

,Periodo,numero_legisladores
0,0,12518
1,1,12328
2,2,8162
3,3,8065
4,4,5853
5,5,5786
6,6,4360
7,7,4338
8,8,3513
9,9,3476
